# 03 — Feature Engineering (VaxFlow)

สร้างฟีเจอร์อนุกรมเวลาต่อ (สาขา × ผลิตภัณฑ์) สำหรับโมเดลพยากรณ์ดีมานด์ 
รวมถึงค่า **Simple Moving Average** และ **Exponential Smoothing** ตามสมการใน Proposal §4.3

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
CLEAN = ROOT / 'data' / 'vaccine' / 'clean'
FEAT = ROOT / 'data' / 'vaccine' / 'features'; FEAT.mkdir(parents=True, exist_ok=True)
demand = pd.read_csv(CLEAN / 'demand_daily.csv', parse_dates=['date'])
demand = demand.sort_values(['hospital_id', 'product_id', 'date']).reset_index(drop=True)

## 1) ฟีเจอร์เวลา + lag + rolling (SMA)

`F(t+1) = (Σ D(t-i+1)) / n` — ค่าเฉลี่ยเคลื่อนที่ใช้เป็นพยากรณ์สภาวะปกติ

In [ ]:
def add_features(g, sma_window=7):
    g = g.copy()
    g['dow'] = g['date'].dt.weekday
    g['is_weekend'] = (g['dow'] >= 5).astype(int)
    for lag in (1, 7, 14):
        g[f'lag_{lag}'] = g['demand'].shift(lag)
    g['sma_7'] = g['demand'].shift(1).rolling(7).mean()        # SMA (พยากรณ์ปกติ)
    g['sma_14'] = g['demand'].shift(1).rolling(14).mean()
    g['roll_std_7'] = g['demand'].shift(1).rolling(7).std()    # ความผันผวน
    return g

feat = demand.groupby(['hospital_id', 'product_id'], group_keys=False).apply(add_features)

## 2) Exponential Smoothing (สภาวะวิกฤต)

`F(t+1) = α·D(t) + (1-α)·F(t)` — ให้น้ำหนักข้อมูลล่าสุดมากกว่า เหมาะกับดีมานด์ที่ผันผวน

In [ ]:
def exp_smoothing(series, alpha=0.4):
    f = np.zeros(len(series)); f[0] = series.iloc[0]
    for t in range(1, len(series)):
        f[t] = alpha * series.iloc[t - 1] + (1 - alpha) * f[t - 1]
    return f

feat['es_0.4'] = (feat.groupby(['hospital_id', 'product_id'])['demand']
                      .transform(lambda s: exp_smoothing(s, 0.4)))
feat = feat.dropna().reset_index(drop=True)
print('features:', feat.shape)
feat.head()

## 3) บันทึกชุดฟีเจอร์

In [ ]:
feat.to_csv(FEAT / 'demand_features.csv', index=False, encoding='utf-8-sig')
print('[saved]', (FEAT / 'demand_features.csv').resolve())
print('columns:', list(feat.columns))